#### 库

In [ ]:
from transformers import AutoTokenizer, EsmForProteinFolding
import torch
import os
import gc

from torch import distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
# from torch.distributed.fsdp.wrap import default_auto_wrap_policy
from torch.distributed.fsdp import MixedPrecision
import multiprocessing

#### 环境配置

In [ ]:
# 内存块设置
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:64"

# 多cpu加载
cpu_count = multiprocessing.cpu_count()
threads_per_proc = max(1, cpu_count)
os.environ["OMP_NUM_THREADS"] = str(threads_per_proc)


def init_dist():
    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank    

# cuda设备号
local_rank = init_dist()

#### 模型设置

In [ ]:
# transformer模型加载
fold_path='../tools/esmfold_v1/'
tokenizer = AutoTokenizer.from_pretrained(fold_path)
model = EsmForProteinFolding.from_pretrained(fold_path, low_cpu_mem_usage=True).eval()

# 参数训练冻结
for p in model.parameters():
    p.requires_grad_(False)

# 模型转移到gpu
device = torch.device(f"cuda:{local_rank}")
model.to(device)
# model.esm = model.esm.half()

# FSDP包装模型
mp = MixedPrecision(param_dtype=torch.float16, reduce_dtype=torch.float16, buffer_dtype=torch.float32)
model.esm = FSDP(
    model.esm,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # 全分片最省显存
    device_id=device,
    use_orig_params=True,
    mixed_precision=mp
)

#### 运行

input_seq = 'LLLLLEEEEEEEEEE
tokenized_input = tokenizer([input_seq], return_tensors="pt", add_special_tokens=False)['input_ids']
tokenized_input = tokenized_input.to(device)
with torch.no_grad():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(tokenized_input)
gc.collect()
torch.cuda.empty_cache()